# Lab 8.4 &mdash; Sequential Pipeline of Specialists

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Wire a StateGraph triage -> billing -> tech with edges
- Run billing and tech as real create_agent specialist nodes
- See how one edge fixes the order and each stage adds to shared state

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Sequential — a pipeline of specialists](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-08-04")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The simplest collaboration is the **sequential pipeline** (deck slide 9): specialists run in a **fixed
order**, each adding to the case &mdash; for support that's **triage &rarr; billing &rarr; tech**. In LangGraph
you express order with **edges**: `add_edge("triage", "billing")` makes billing run *after* triage. Each stage
reads shared state and appends its finding, so a later stage sees everything so far. Here `triage` is
deterministic and `billing`/`tech` are **real `create_agent` specialists** &mdash; a real pipeline of agents.
(Trade-off: it's serial, so latency adds up, and an early error flows downstream.)

In [ ]:
# Each stage is a node; edges fix the order. billing/tech are real create_agent specialists.
print("pipeline graph: triage -> billing(real) -> tech(real)")

## Build it
Complete the **edges** that chain `triage -> billing -> tech`. The triage node is written; `billing` and
`tech` are real specialist nodes.

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id: str) -> str:
    """Look up the charges on an order by its id, e.g. '4471'. Use for billing / charge / refund questions."""
    charges = INVOICES.get(order_id.strip(), [])
    return str(charges) if charges else "no charges found for that order"

@tool
def known_issues(symptom: str) -> str:
    """Look up a known technical issue by symptom keyword, e.g. 'crash' or 'login'. Use for tech problems."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v["bug"] + ": " + v["fix"]
    return "no known issue matched"

class CaseState(TypedDict):
    message: str
    findings: Annotated[list, add]

def build_specialist(tools, role):
    return create_agent(llm, tools, system_prompt=f"You are the {role} specialist. Use ONLY your own tools; answer in one sentence.")

def specialist_node(role, tools):
    """Wrap a REAL create_agent specialist as a pipeline stage."""
    agent = build_specialist(tools, role)
    def node(state):
        r = agent.invoke({"messages": [("user", state["message"])]}, config={"recursion_limit": 8})
        return {"findings": [f"{role}: " + r["messages"][-1].content]}
    return node

def triage_node(state):
    """Stage 1 (deterministic): sort the ticket before the specialists see it."""
    return {"findings": ["triage: billing + tech needed"]}

# Workflow (StateGraph) -- specialists run in a fixed order, each adding to the case:
#
#   START -> triage -> billing -> tech -> END   (billing/tech are real create_agent nodes)
def build_pipeline():
    g = StateGraph(CaseState)
    g.add_node("triage", triage_node)
    g.add_node("billing", specialist_node("billing", [lookup_invoice]))
    g.add_node("tech", specialist_node("tech", [known_issues]))
    g.add_edge(START, "triage")
    ___   # TODO: chain the stages in order -- triage -> billing, then billing -> tech
    g.add_edge("tech", END)
    return g.compile()

try:
    # Offline sanity (no model call): the deterministic triage stage + the intended order.
    print("triage stage:", triage_node({"message": "charged twice, app crashing"}))
    print("pipeline order: START -> triage -> billing -> tech -> END")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build the graph and run one ticket through it. Each stage adds its finding to shared state in the wired order.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        pipe = build_pipeline()
        print("graph nodes:", sorted(set(pipe.get_graph().nodes) - {"__start__", "__end__"}))

        final = pipe.invoke(
            {"message": "I was charged twice for order 4471 and the app keeps crashing on login.", "findings": []},
            config={"recursion_limit": 12})
        print("\ncase, stage by stage:")
        for f in final["findings"]:
            print("  •", f)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The `add_edge` chain fixes the **order**: triage, then billing, then tech &mdash; each a real node appending to shared `findings`.
- Swap or insert a stage by rewiring **one edge** &mdash; the framework sequences the team, not a bespoke loop.
- Because it's serial, latency is the **sum** of stages and an early error propagates downstream &mdash; the price of a clean ordered hand-off. Fan-out (Lab 8.5) trades that for parallelism.

## Your turn (open task &mdash; no grader)
Insert a **`policy`** node between billing and tech (a deterministic stage, e.g. returns
`"policy: refund within 30 days"`) and rewire the two edges around it. **What good looks like:** the new stage runs
in order, every stage's finding is in the final state, and removing it cleanly shortens the graph &mdash; no other
node changes.

---
### Key takeaway
A sequential pipeline is a StateGraph edge-chain: add_edge fixes the order and each node -- a real specialist -- adds to shared state. Clean ordered hand-offs, at the cost of serial latency. Next: run specialists in parallel.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>